# Whisper-medium CV-mn pipeline smoke test

Runs every stage of the training pipeline on a tiny subset so you can verify the local dataset, processor, model, collator, forward pass, and `generate()` all work before launching a full run on the RTX 5080.

Run from the repo root (`script_training/`).

In [ ]:
import os, sys, pathlib

REPO = pathlib.Path().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
SRC = REPO / 'src'
sys.path.insert(0, str(SRC))
os.chdir(REPO)
print('repo:', REPO)
print('src on path:', SRC.exists())

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('bf16 supported:', torch.cuda.is_bf16_supported())

## 1. Local dataset loads and splits

In [ ]:
from local_dataset import load_common_voice_mn

DATA_ROOT = 'lab3/common_voice_mn'
ds = load_common_voice_mn(DATA_ROOT, sampling_rate=16000, val_ratio=0.05, test_ratio=0.05, seed=42)
for k, v in ds.items():
    print(f'{k}: {len(v)}')
ds['train'][0]

## 2. Processor loads and Mongolian tokens round-trip

In [ ]:
from transformers import WhisperProcessor
MODEL_ID = 'openai/whisper-medium'
processor = WhisperProcessor.from_pretrained(MODEL_ID, language='mn', task='transcribe')

sample_text = ds['train'][0]['transcription']
ids = processor.tokenizer(sample_text).input_ids
decoded = processor.tokenizer.decode(ids, skip_special_tokens=True)
print('sample:', sample_text)
print('decoded:', decoded)
assert sample_text.strip() in decoded or decoded.strip() in sample_text, 'tokenizer round-trip mismatch'

## 3. Preprocessing + filtering on a tiny subset

In [ ]:
from argparse import Namespace
from train import load_and_prepare_datasets

args = Namespace(
    data_root=DATA_ROOT,
    val_ratio=0.05,
    test_ratio=0.05,
    seed=42,
    debug=True,
    debug_subset_size=16,
    num_workers=1,
    max_input_length=20.0,
)
small = load_and_prepare_datasets(args, processor)
for k, v in small.items():
    print(f'{k}: {len(v)} | columns: {v.column_names}')
small['train'][0].keys()

## 4. Collator builds a batch

In [ ]:
from data_collate import DataCollatorSpeechSeq2SeqWithPadding
collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=processor.tokenizer.bos_token_id,
)
features = [small['train'][i] for i in range(min(4, len(small['train'])))]
batch = collator(features)
print({k: tuple(v.shape) for k, v in batch.items()})
assert batch['input_features'].ndim == 3
assert batch['labels'].ndim == 2

## 5. Model loads, forward + generate work

In [ ]:
args2 = Namespace(
    model_name_or_path=MODEL_ID,
    use_lora=True,
    lora_r=32,
    lora_alpha=64,
    gradient_checkpointing=False,
)
from train import setup_model
model = setup_model(args2, processor)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print('model on', device)

In [ ]:
batch_dev = {k: v.to(device) for k, v in batch.items()}
out = model(**batch_dev)
print('loss:', float(out.loss))
assert torch.isfinite(out.loss), 'loss is not finite'

In [ ]:
with torch.no_grad():
    gen = model.generate(
        input_features=batch_dev['input_features'],
        max_new_tokens=64,
        num_beams=1,
        language='mn',
        task='transcribe',
    )
pred = processor.batch_decode(gen, skip_special_tokens=True)
labels = batch_dev['labels'].clone()
labels[labels == -100] = processor.tokenizer.pad_token_id
refs = processor.batch_decode(labels, skip_special_tokens=True)
for p, r in zip(pred, refs):
    print('PRED:', p)
    print('REF :', r)
    print('---')

## 6. WER metric + normalizer

In [ ]:
import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
wer = evaluate.load('wer')
norm = BasicTextNormalizer()
p_n = [norm(x) for x in pred]
r_n = [norm(x) for x in refs]
pairs = [(p, r) for p, r in zip(p_n, r_n) if r.strip()]
if pairs:
    p_n, r_n = zip(*pairs)
    print('smoke WER (untrained, meaningless):', wer.compute(predictions=list(p_n), references=list(r_n)))

## 7. One optimizer step (end-to-end sanity)

In [ ]:
trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=1e-4)
model.train()
out = model(**batch_dev)
out.loss.backward()
opt.step(); opt.zero_grad()
print('one optimizer step OK, loss =', float(out.loss))

If every cell above ran cleanly, launch the full run:

```bash
python src/main.py train \
  --data_root lab3/common_voice_mn \
  --model_name_or_path openai/whisper-medium \
  --num_train_epochs 10 --train_batch_size 8 --gradient_accumulation_steps 4 \
  --learning_rate 1e-4 --mixed_precision bf16 \
  --output_dir ./whisper-medium-mn-lora
```